# Этап 5: Ансамбль и объяснимость (SHAP)

В этом ноутбуке:
1. Создаем ансамбль из Random Forest и BERT
2. Используем SHAP для объяснения Random Forest
3. Анализируем ошибки моделей

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

import shap

from train_rf import RandomForestTrainer
from train_bert import BERTTrainer

sns.set_style('whitegrid')
%matplotlib inline

## 5.1 Загрузка моделей

In [ ]:
# Random Forest
rf_trainer = RandomForestTrainer()
rf_trainer.load_model('../models/rf_model.pkl')

# BERT
bert_trainer = BERTTrainer()
bert_trainer.load_model('../models/bert_model')

print("✓ Модели загружены")

## 5.2 Подготовка тестовых данных

In [ ]:
# Загрузка данных с признаками
df_features = pd.read_csv('../data/features/features.csv')
df_labeled = pd.read_json('../data/processed/reviews_labeled.json')

# Подготовка для RF
X, y = rf_trainer.prepare_features(df_features)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Тексты для BERT
texts_train, texts_test, _, _ = train_test_split(
    df_labeled['text'].values, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Test size: {len(X_test)}")

## 5.3 Создание ансамбля

In [ ]:
# Предсказания Random Forest
rf_proba = rf_trainer.model.predict_proba(X_test)[:, 1]

# Предсказания BERT
bert_proba = bert_trainer.predict(texts_test)

# Ансамбль (простое усреднение)
ensemble_proba = (rf_proba + bert_proba) / 2
ensemble_pred = (ensemble_proba > 0.5).astype(int)

print("✓ Предсказания получены")

## 5.4 Сравнение моделей

In [ ]:
# Метрики
rf_auc = roc_auc_score(y_test, rf_proba)
bert_auc = roc_auc_score(y_test, bert_proba)
ensemble_auc = roc_auc_score(y_test, ensemble_proba)

print("=== Сравнение моделей ===")
print(f"Random Forest ROC-AUC: {rf_auc:.4f}")
print(f"BERT ROC-AUC: {bert_auc:.4f}")
print(f"Ансамбль ROC-AUC: {ensemble_auc:.4f}")

print("\n=== Classification Report (Ансамбль) ===")
print(classification_report(y_test, ensemble_pred, target_names=['Real', 'Fake']))

In [ ]:
# Визуализация сравнения
comparison_df = pd.DataFrame({
    'Model': ['Random Forest', 'BERT', 'Ensemble'],
    'ROC-AUC': [rf_auc, bert_auc, ensemble_auc]
})

plt.figure(figsize=(10, 6))
plt.bar(comparison_df['Model'], comparison_df['ROC-AUC'], color=['steelblue', 'coral', 'green'])
plt.ylabel('ROC-AUC Score')
plt.title('Сравнение моделей')
plt.ylim([0.5, 1.0])
for i, v in enumerate(comparison_df['ROC-AUC']):
    plt.text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 5.5 SHAP для Random Forest

Используем SHAP для объяснения предсказаний Random Forest

In [ ]:
# Создаем SHAP explainer
explainer = shap.TreeExplainer(rf_trainer.model)

# Вычисляем SHAP values (используем подвыборку для скорости)
X_test_sample = X_test.sample(min(100, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_test_sample)

print("✓ SHAP values вычислены")

In [ ]:
# SHAP Summary Plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values[1], X_test_sample, show=False, max_display=15)
plt.tight_layout()
plt.savefig('../reports/figures/shap_summary.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Bar Plot (средняя важность)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values[1], X_test_sample, plot_type='bar', show=False, max_display=15)
plt.tight_layout()
plt.savefig('../reports/figures/shap_bar.png', dpi=300, bbox_inches='tight')
plt.show()

## 5.6 Анализ конкретных примеров

In [ ]:
# Выбираем интересные примеры
# 1. Правильно классифицированный фейк
fake_idx = X_test_sample[y_test[X_test_sample.index] == 1].index[0]

# SHAP Force Plot
shap.initjs()
idx_in_sample = X_test_sample.index.get_loc(fake_idx)
shap.force_plot(
    explainer.expected_value[1],
    shap_values[1][idx_in_sample],
    X_test_sample.iloc[idx_in_sample],
    matplotlib=True,
    show=False
)
plt.tight_layout()
plt.savefig('../reports/figures/shap_force_fake.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Пример фейкового отзыва (индекс {fake_idx}):")
print(df_labeled.loc[fake_idx, 'text'])

## 5.7 Анализ ошибок

In [ ]:
# Находим ошибки ансамбля
errors_df = pd.DataFrame({
    'true_label': y_test,
    'pred_label': ensemble_pred,
    'rf_proba': rf_proba,
    'bert_proba': bert_proba,
    'ensemble_proba': ensemble_proba,
    'text': texts_test
})

errors_df['is_error'] = errors_df['true_label'] != errors_df['pred_label']

print(f"Всего ошибок: {errors_df['is_error'].sum()} ({errors_df['is_error'].mean()*100:.1f}%)")

# False Positives (реальный классифицирован как фейк)
fp = errors_df[(errors_df['true_label'] == 0) & (errors_df['pred_label'] == 1)]
print(f"\nFalse Positives: {len(fp)}")

# False Negatives (фейк классифицирован как реальный)
fn = errors_df[(errors_df['true_label'] == 1) & (errors_df['pred_label'] == 0)]
print(f"False Negatives: {len(fn)}")

In [ ]:
# Примеры ошибок
print("=== Примеры False Positives (реальные, но классифицированы как фейк) ===")
for idx, row in fp.head(3).iterrows():
    print(f"\nТекст: {row['text'][:150]}...")
    print(f"RF: {row['rf_proba']:.3f}, BERT: {row['bert_proba']:.3f}, Ансамбль: {row['ensemble_proba']:.3f}")

print("\n=== Примеры False Negatives (фейки, но классифицированы как реальные) ===")
for idx, row in fn.head(3).iterrows():
    print(f"\nТекст: {row['text'][:150]}...")
    print(f"RF: {row['rf_proba']:.3f}, BERT: {row['bert_proba']:.3f}, Ансамбль: {row['ensemble_proba']:.3f}")

## 5.8 Выводы

**Результаты:**
1. Ансамбль показывает лучшие результаты, чем отдельные модели
2. Random Forest хорошо работает на структурированных признаках
3. BERT лучше понимает семантику текста

**SHAP анализ показал:**
- Самые важные признаки: text_length, fake_keywords_count, rating
- Короткие тексты с высоким рейтингом чаще классифицируются как фейк
- Наличие конкретики (цифры, детали) снижает вероятность фейка

**Ошибки модели:**
- False Positives: Короткие позитивные реальные отзывы
- False Negatives: Длинные детальные фейковые отзывы